# Capstone G5: Predicting heart disease from a checkup

Thirteen numbers from a routine heart checkup -- cholesterol, blood pressure, age, chest-pain type, an exercise test -- used to flag heart disease. A classic prediction task on 303 patients.

**Your group's priority: fairness across sex.** Heart disease shows up differently in women and has long been under-diagnosed in them. A model trained on mostly-male data can be accurate overall yet worse for women.

**Your goal (the rubric):** build it, check honestly how well it works, find a case where it fails, defend one choice you made, and make sure *both* partners can explain everything.

### How to use this notebook (read this first)

- A notebook is a stack of **cells**. A gray cell is **code**; this white cell is a note.
- To run a code cell, click it and press **Shift+Enter**. Run them **top to bottom, in order** --
  later cells use things earlier cells made.
- **Red text is normal**, not a disaster -- it's usually a warning. A real error stops with a
  message; read the last line, or paste it to Claude and ask "what does this mean?"
- You will not understand every line, and that's fine. But you should be able to say, in plain
  English, *what each cell is for*. Stuck on a line? Ask Claude: **"explain this cell line by line."**
- Nothing you click here can break anything. Experiment.

### You have Claude Code -- so here's your *real* job

You can already make a computer write code: just describe what you want to Claude Code and it will
build it. So **writing code is no longer the hard part.** The skill that actually makes you good at
this -- the thing this notebook is training -- is **judgment**:

1. **What are you trying to build?** "Accurate" isn't a goal. A cancer screen that must never miss a
   tumor is a *different tool* than one that's just right-on-average. Decide what "good" means first.
2. **What tools exist?** Different models, different ways to prep the data, different ways to grade a
   model. This notebook is a guided tour of that toolbox.
3. **Why does each tool exist** -- what problem it was invented to solve.
4. **How do you pick the right one?** That's the whole game. The wrong tool gives a wrong answer even
   with flawless code.

So: read the notes, play with the dials, and when you want to change or add something, **describe it
to Claude Code and let it write the code** -- then be ready to explain *why you chose it*. That "why"
is what you're graded on, and it's what a real scientist or doctor would ask you.

In [ ]:
# WHAT THIS DOES: gets the course files onto this Colab machine and installs the tools we need.
# (On your own computer it does nothing.) Just run it once and wait for "setup ok".
import os, sys
if not os.path.exists("../../capstone_common.py"):
    os.system("git clone -q https://github.com/jinchiwei/outset-ai-healthcare.git")
    os.chdir("outset-ai-healthcare/notebooks/day3_capstone/projects/g5_heart")
sys.path.insert(0, "../..")            # so we can import the course helper files
sys.path.insert(0, "../../../_shared")
import colab_setup
colab_setup.ensure(*colab_setup.DAY3_TABULAR)

### Plain-English vocabulary

- This is **tabular** data -- a **spreadsheet**, one row per patient, one column per measurement.
- The columns you use as clues are **features** (age, cholesterol, ...). The column you're trying to
  predict is the **target** (does this person have the disease -- yes/no).
- A **model** finds a rule linking the features to the target. We test it on patients it never saw,
  because a model that *memorized* the training rows would look smart and be useless on new people.

In [ ]:
# WHAT THIS DOES: loads the spreadsheet and shows the first few rows.
import sys
sys.path.insert(0, "../..")
import capstone_tabular as ct

df, meta = ct.load_heart()
print(meta["name"], "  ->", df.shape[0], "patients,", len(meta["features"]), "features")
print("features (the clues):", meta["features"])
print("predicting:", meta["target"], f"(1 = {meta['positive']})")
df.head()   # each row = one person; each column = one measurement

### Look before you model
**What this does:** counts how many people fall in each class and summarizes each feature. Ask: is it balanced? Could two features be tangled together?

In [ ]:
import matplotlib.pyplot as plt
print(df[meta["target"]].value_counts().rename({0: "no", 1: "yes"}).to_string())
df[meta["features"]].describe().T[["mean", "min", "max"]]   # typical / smallest / largest value per feature

### Build your own model (interactive)

**What this does:** you pick which **features** the model may look at and which **model** to use, then
it reports the test accuracy. **Try dropping features** -- does accuracy hold with fewer? That tells
you which measurements actually carry the signal (and which are just along for the ride).

**How to choose a model:** start with **Logistic Regression** -- it's simple and you can explain
exactly why it decided what it did (gold in medicine). Only reach for Random Forest / CatBoost /
TabPFN if the simple one plateaus *and* you suspect the clues interact in tangled ways. On a small
spreadsheet, fancier often just overfits. Simplest tool that does the job wins.

In [ ]:
from ipywidgets import interact_manual, SelectMultiple, Dropdown

def build(features, model):
    if not features:
        print("Pick at least one feature."); return
    ct.fit_eval(df, list(features), meta["target"], model=model)

try:
    interact_manual(build,
        features=SelectMultiple(options=meta["features"], value=tuple(meta["features"]),
                                description="features", rows=min(10, len(meta["features"])),
                                style={"description_width": "initial"}),
        model=Dropdown(options=list(ct.make_models()), value="Logistic Regression", description="model"))
except ImportError:
    ct.fit_eval(df, meta["features"], meta["target"])

### Fairness: does it work equally well for everyone?

Your priority is **fairness across sex**. **What this does:** trains once, then reports the test accuracy
**separately for each group** (women vs men).

**Why this matters -- the analogy:** a restaurant can average 4.5 stars while giving every vegetarian
food poisoning. An "average" hides the group it fails. A medical model that's accurate overall but
worse for one group is exactly that restaurant.

In [ ]:
from ipywidgets import interact_manual, Dropdown

def fairness(model):
    ct.audit_by_group(df, meta["features"], meta["target"],
                      meta["group"], meta.get("group_names"), model=model)

try:
    interact_manual(fairness,
        model=Dropdown(options=list(ct.make_models()), value="Logistic Regression", description="model"))
except ImportError:
    ct.audit_by_group(df, meta["features"], meta["target"], meta["group"], meta.get("group_names"))

### One strong approach (peek only if you're stuck)

*This is roughly what the worked solution does. It's a nudge, not the code -- describe these steps to
Claude and build them yourself, then be ready to explain why.*

**Bake off a few models** (logistic regression, random forest, CatBoost, TabPFN) and let the AUC pick the winner. Then ask **which clues actually matter**: compare a cholesterol-only model to the full one (cholesterol alone is surprisingly weak), and use **SHAP** to see the real drivers. Finally, **audit accuracy separately for women and men** -- heart disease is historically under-diagnosed in women, and this data skews male.

### Where to take it (ask Claude to help)
- Does **cholesterol alone** predict much? Compare a cholesterol-only model to the full one -- you may be surprised which clues carry the signal.
- The data is about 2-to-1 male. Could that explain any accuracy gap by sex? What would you collect to fix it?
- Turn the output into a risk *percentage* instead of yes/no -- how would a doctor use that?